# Test Generation Pipeline

Notebook ini menguji pipeline generation KUHP RAG:
1. **Unit test** — parsing verifikasi, format konteks, sitasi, dan gate answer-if-supported dengan LLM tiruan (tanpa Ollama)
2. **Integrasi** — retrieval + generation + verifikasi end-to-end (butuh Ollama)

Jalankan cell **Unit Test** dulu (cepat, tanpa model). Kalau lolos, lanjut ke **Integrasi**.

---
## 1. Unit Test (tanpa LLM)

In [ ]:
import sys; sys.path.insert(0, "src")
from langchain_core.documents import Document
from generation import (
    format_context, extract_cited_pasal, parse_verification,
    VerificationResult, GenerationResult, format_answer,
    generate,
)
from generation_prompts import (
    generate_messages, verify_messages,
    PESAN_ABSTAIN, PESAN_TIDAK_ADA_KONTEKS,
)
print("Import OK")

In [ ]:
# --- Test format_context ---
docs = [
    Document(id="476", page_content="Setiap Orang yang mengambil suatu Barang...", metadata={"pasal": 476}),
    Document(id="477", page_content="Pencurian dengan kekerasan...", metadata={"pasal": 477}),
]
ctx = format_context(docs)
assert "[Pasal 476]" in ctx
assert "[Pasal 477]" in ctx
assert "Setiap Orang" in ctx
print("✅ format_context")
print(ctx)

In [ ]:
# --- Test extract_cited_pasal ---
assert extract_cited_pasal("Menurut [Pasal 476] dan [Pasal 477], ...") == [476, 477]
assert extract_cited_pasal("Tidak ada sitasi di sini") == []
assert extract_cited_pasal("[Pasal 1] [Pasal 1] duplikat") == [1]
assert extract_cited_pasal("[Pasal 300], [Pasal 100], [Pasal 200]") == [100, 200, 300]
# format sitasi yang benar-benar muncul di keluaran model
assert extract_cited_pasal("[Pasal 300 huruf a]") == [300]
assert extract_cited_pasal("[Pasal 476 ayat (1)]") == [476]
assert extract_cited_pasal("[Pasal 476, Pasal 477]") == [476, 477]
# penyebutan di luar kurung siku (rujukan silang) bukan sitasi
assert extract_cited_pasal("sebagaimana dimaksud dalam Pasal 100 [Pasal 476]") == [476]
print("✅ extract_cited_pasal")

In [ ]:
# --- Test parse_verification ---

# Kasus 1: semua didukung
v1 = parse_verification(
    "DIDUKUNG: ya\nKLAIM TIDAK DIDUKUNG: tidak ada\nPASAL YANG DIRUJUK: 476, 477"
)
assert v1.supported == "ya"
assert v1.is_supported == True
assert v1.cited_pasal == [476, 477]
assert v1.unsupported_claims == []
print("✅ parse_verification: semua didukung")

# Kasus 2: sebagian didukung → TIDAK lolos gate (direvisi atau di-abstain, tidak ditampilkan utuh)
v2 = parse_verification(
    "DIDUKUNG: sebagian\nKLAIM TIDAK DIDUKUNG: Hukuman 10 tahun tidak disebutkan\nPASAL YANG DIRUJUK: 476"
)
assert v2.supported == "sebagian"
assert v2.is_supported == False
assert len(v2.unsupported_claims) == 1
print("✅ parse_verification: sebagian → tidak lolos")

# Kasus 3: tidak didukung
v3 = parse_verification(
    "DIDUKUNG: tidak\nKLAIM TIDAK DIDUKUNG: Semua klaim salah\nPASAL YANG DIRUJUK: -"
)
assert v3.supported == "tidak"
assert v3.is_supported == False
print("✅ parse_verification: tidak didukung")

# Kasus 4: output kosong / gagal parse → default "tidak" (aman)
v4 = parse_verification("")
assert v4.is_supported == False
print("✅ parse_verification: output kosong → default tidak")

# Kasus 5: huruf tebal markdown tetap terbaca (dulu terbaca "tidak" → jawaban benar ditolak)
v5 = parse_verification("**DIDUKUNG:** ya\n**KLAIM TIDAK DIDUKUNG:** tidak ada\n**PASAL YANG DIRUJUK:** 476")
assert v5.supported == "ya" and v5.cited_pasal == [476]
print("✅ parse_verification: markdown bold")

# Kasus 6: baris DIDUKUNG hilang → tidak boleh dibaca dari baris "KLAIM TIDAK DIDUKUNG"
v6 = parse_verification("KLAIM TIDAK DIDUKUNG: tidak ada\nPASAL YANG DIRUJUK: 476")
assert v6.supported == "tidak" and not v6.is_supported
print("✅ parse_verification: tanpa verdict → tidak")

# Kasus 7: "ya" tetapi menyebut klaim bermasalah → diturunkan jadi "sebagian"
v7 = parse_verification("DIDUKUNG: ya\nKLAIM TIDAK DIDUKUNG: ganti rugi Rp 500 juta\nPASAL YANG DIRUJUK: 476")
assert v7.supported == "sebagian" and not v7.is_supported
print("✅ parse_verification: ya + klaim bermasalah → sebagian")

# Kasus 8: berbagai tulisan "tidak ada klaim bermasalah"
for kosong in ["tidak ada", "Tidak ada.", "-", "(tidak ada)", "Nihil"]:
    v = parse_verification(f"DIDUKUNG: ya\nKLAIM TIDAK DIDUKUNG: {kosong}\nPASAL YANG DIRUJUK: 476")
    assert v.supported == "ya" and v.unsupported_claims == [], kosong
print("✅ parse_verification: variasi 'tidak ada'")

In [ ]:
# --- Test generate() tanpa konteks → langsung abstain ---
result = generate("apa hukuman pencurian?", [], verify=False)
assert result.abstained == True
assert result.status == "abstain"
assert result.is_supported == False
assert PESAN_TIDAK_ADA_KONTEKS in result.answer
print("✅ generate tanpa konteks → abstain")
print(f"   answer: {result.answer[:80]}...")

In [ ]:
# --- Test prompt messages format ---
msgs = generate_messages("apa hukuman pencurian?", "[Pasal 476]\nSetiap Orang...")
assert len(msgs) == 2
assert msgs[0]["role"] == "system"
assert msgs[1]["role"] == "user"
assert "apa hukuman pencurian?" in msgs[1]["content"]
assert "[Pasal 476]" in msgs[1]["content"]
print("✅ generate_messages format")

vmsgs = verify_messages("Jawaban test", "[Pasal 476]\nSetiap Orang...")
assert len(vmsgs) == 2
assert vmsgs[0]["role"] == "system"
assert "Jawaban test" in vmsgs[1]["content"]
print("✅ verify_messages format")

# --- Revisi: umpan balik masuk ke prompt; penanda tercantum di aturan generator ---
from generation_prompts import PENANDA_TIDAK_CUKUP, PENANDA_KLARIFIKASI
msgs_revisi = generate_messages("q", "[Pasal 476]\nX", feedback="- ganti rugi Rp 500 juta tidak disebut")
assert "ganti rugi Rp 500 juta" in msgs_revisi[1]["content"]
assert "ganti rugi" not in generate_messages("q", "ctx")[1]["content"]
assert PENANDA_TIDAK_CUKUP in msgs_revisi[0]["content"] and PENANDA_KLARIFIKASI in msgs_revisi[0]["content"]
print("✅ generate_messages: revisi + penanda")

In [ ]:
# --- Test format_answer ---
result_ok = GenerationResult(
    query="apa hukuman pencurian?",
    answer="Menurut [Pasal 476], pencurian diancam pidana penjara paling lama 5 tahun.",
    cited_pasal=[476],
    is_supported=True,
    verification=VerificationResult(supported="ya", cited_pasal=[476]),
    context_docs=docs,
)
output = format_answer(result_ok)
assert "Pasal 476" in output
assert "Dasar hukum:" in output
print("✅ format_answer (supported)")
print(output)
print()

result_abstain = GenerationResult(
    query="test",
    answer="Mohon maaf...",
    cited_pasal=[],
    is_supported=False,
    verification=VerificationResult(supported="tidak"),
    context_docs=[],
    abstained=True,
)
output2 = format_answer(result_abstain)
assert "Dasar hukum:" not in output2  # abstain → tidak ada sitasi
print("✅ format_answer (abstained)")

---
## 1b. Gate answer-if-supported (LLM tiruan, tanpa Ollama)

Menguji alur generate → verifikasi → revisi → abstain/clarify. LLM diganti `FakeListChatModel` yang mengembalikan jawaban berurutan, jadi yang diuji adalah logika gate, bukan kualitas model.

In [ ]:
# --- LLM tiruan ---
import generation as gen
from langchain_core.runnables import RunnableLambda

DOK = [Document(
    id="476", metadata={"pasal": 476},
    page_content="Pasal 476\n\nSetiap Orang yang mengambil suatu Barang yang sebagian atau seluruhnya milik orang lain, "
                 "dengan maksud untuk dimiliki secara melawan hukum, dipidana karena pencurian dengan pidana penjara "
                 "paling lama 5 (lima) tahun atau pidana denda paling banyak kategori V.",
)]

_ASLI = (gen.get_llm_generate, gen.get_llm_verify)

def tiruan(jawaban):
    """Runnable pengganti LLM: mengembalikan jawaban berurutan (yang terakhir diulang) dan mencatat tiap
    panggilan. Mengembalikan (runnable, catatan): catatan["n"] = jumlah panggilan, catatan["pesan"] = pesan
    yang diterima pada tiap panggilan."""
    catatan = {"n": 0, "pesan": []}
    def llm(pesan):
        teks = jawaban[min(catatan["n"], len(jawaban) - 1)]
        catatan["n"] += 1
        catatan["pesan"].append(pesan)
        return teks
    return RunnableLambda(llm), catatan

def pakai(jawaban, verifikasi=None):
    """Ganti LLM generate/verify. Mengembalikan (catatan_generate, catatan_verify)."""
    llm_g, cat_g = tiruan(jawaban)
    llm_v, cat_v = tiruan(verifikasi or [YA])
    gen.get_llm_generate = lambda: llm_g
    gen.get_llm_verify = lambda: llm_v
    return cat_g, cat_v

def pulihkan():
    gen.get_llm_generate, gen.get_llm_verify = _ASLI

JAWAB_BAIK = "Pencurian dipidana penjara paling lama 5 tahun atau denda kategori V [Pasal 476]."
JAWAB_NGARANG = JAWAB_BAIK + " Pelaku juga wajib membayar ganti rugi Rp 500 juta [Pasal 476]."
SEBAGIAN = "DIDUKUNG: sebagian\nKLAIM TIDAK DIDUKUNG: ganti rugi Rp 500 juta tidak disebut\nPASAL YANG DIRUJUK: 476"
YA = "DIDUKUNG: ya\nKLAIM TIDAK DIDUKUNG: tidak ada\nPASAL YANG DIRUJUK: 476"
print("Setup LLM tiruan OK")

In [ ]:
# 1. Verdict "ya" → jawaban + sitasi, tanpa revisi; generator dan verifikator masing-masing dipanggil sekali
cat_g, cat_v = pakai([JAWAB_BAIK], [YA])
r = generate("hukuman mencuri?", DOK)
assert r.status == "jawab" and r.is_supported and not r.abstained
assert r.cited_pasal == [476] and r.attempts == 1 and cat_g["n"] == 1 and cat_v["n"] == 1
print("✅ verdict ya → jawab")

# 2. Verdict "sebagian" → revisi → "ya": klaim yang ditolak tidak tampil
cat_g, cat_v = pakai([JAWAB_NGARANG, JAWAB_BAIK], [SEBAGIAN, YA])
r = generate("hukuman mencuri?", DOK)
assert r.status == "jawab" and r.attempts == 2 and cat_g["n"] == 2 and cat_v["n"] == 2
assert "Rp 500 juta" not in r.answer
# alasan penolakan benar-benar sampai ke generator pada percobaan kedua (dan tidak ada di percobaan pertama)
percobaan1, percobaan2 = cat_g["pesan"][0][1]["content"], cat_g["pesan"][1][1]["content"]
assert "Jawaban sebelumnya" not in percobaan1
assert "Jawaban sebelumnya" in percobaan2 and "ganti rugi Rp 500 juta tidak disebut" in percobaan2
print("✅ sebagian → revisi (umpan balik sampai ke generator) → jawab")

# 3. "sebagian" terus → abstain; klaim yang gagal TIDAK ikut tampil (hanya daftar pasal hasil retrieval)
cat_g, cat_v = pakai([JAWAB_NGARANG], [SEBAGIAN])
r = generate("hukuman mencuri?", DOK)
assert r.status == "abstain" and r.abstained and not r.is_supported and r.cited_pasal == []
assert r.attempts == 2 and cat_g["n"] == 2
assert "Rp 500 juta" not in r.answer and "Pasal 476:" in r.answer
print("✅ sebagian terus → abstain tanpa teks LLM")

# 4. max_revisions=0 → abstain pada kegagalan pertama, generator dipanggil sekali
cat_g, _ = pakai([JAWAB_NGARANG, JAWAB_BAIK], [SEBAGIAN, YA])
r = generate("hukuman mencuri?", DOK, max_revisions=0)
assert r.status == "abstain" and cat_g["n"] == 1
print("✅ max_revisions=0")

In [ ]:
# 5. Sitasi di luar konteks → ditolak deterministik pada KEDUA percobaan, verifikator LLM tidak pernah dipanggil
cat_g, cat_v = pakai(["Pencurian dipidana 5 tahun [Pasal 999]."], [YA])
r = generate("hukuman mencuri?", DOK)
assert r.status == "abstain" and cat_g["n"] == 2 and cat_v["n"] == 0 and 999 not in r.cited_pasal
print("✅ sitasi di luar konteks → ditolak tanpa LLM")

# 6. Tanpa sitasi sama sekali → ditolak
cat_g, cat_v = pakai(["Pencurian dipidana penjara 5 tahun."], [YA])
r = generate("hukuman mencuri?", DOK)
assert r.status == "abstain" and cat_v["n"] == 0
print("✅ tanpa sitasi → ditolak")

# 7. Sitasi yang ditampilkan diambil dari jawaban, bukan dari baris "PASAL YANG DIRUJUK" verifikator
pakai([JAWAB_BAIK], ["DIDUKUNG: ya\nKLAIM TIDAK DIDUKUNG: tidak ada\nPASAL YANG DIRUJUK: 999"])
r = generate("hukuman mencuri?", DOK)
assert r.status == "jawab" and r.cited_pasal == [476]
print("✅ sitasi diambil dari jawaban")

# 8. Rujukan silang di luar kurung siku bukan sitasi
pakai(["Pidana sebagaimana dimaksud dalam Pasal 100 [Pasal 476]."], [YA])
r = generate("hukuman mencuri?", DOK)
assert r.status == "jawab" and r.cited_pasal == [476]
print("✅ rujukan silang tidak dianggap sitasi")

In [ ]:
# 9. Penanda [BUKTI_TIDAK_CUKUP] → abstain di percobaan pertama, verifikator tidak dipanggil, tanpa "Dasar hukum"
cat_g, cat_v = pakai(["[BUKTI_TIDAK_CUKUP] Konteks hanya mengatur pencurian biasa."], [YA])
r = generate("bagaimana kalau pura-pura jadi polisi?", DOK)
assert r.status == "abstain" and r.abstained and r.cited_pasal == []
assert cat_g["n"] == 1 and cat_v["n"] == 0
assert "Dasar hukum" not in format_answer(r)
print("✅ penanda tidak cukup → abstain")

# 10. Penanda [KLARIFIKASI] → pertanyaan balik (cabang Clarify)
cat_g, cat_v = pakai(["[KLARIFIKASI] Apakah barangnya diambil dengan kekerasan atau tanpa sepengetahuan pemiliknya?"], [YA])
r = generate("barang saya diambil orang", DOK)
assert r.status == "klarifikasi" and r.abstained and r.answer.startswith("Apakah barangnya")
assert cat_v["n"] == 0
print("✅ penanda klarifikasi → clarify")

# 11. verify=False: pemeriksaan dilewati (verifikator 0 panggilan), tetap membaca sitasi berbagai bentuk
cat_g, cat_v = pakai(["Dipidana penjara 5 tahun [Pasal 476 ayat (1)]."], [YA])
r = generate("hukuman mencuri?", DOK, verify=False)
assert r.status == "jawab" and r.cited_pasal == [476] and cat_v["n"] == 0
print("✅ verify=False")

pulihkan()

In [ ]:
print("\n" + "="*50)
print("Semua unit test LULUS ✅")
print("="*50)
print("\nLanjut ke bagian Integrasi (butuh Ollama running).")

---
## 2. Integrasi (butuh Ollama + model + data)

In [ ]:
import torch  # harus paling atas (lihat catatan macOS)
import sys, time; sys.path.insert(0, "src")
from retrieval import load_resources, search
from generation import generate, format_answer, ask

print("Loading resources...")
t0 = time.time()
res = load_resources()
print(f"Loaded {res.n} pasal dalam {time.time()-t0:.1f} detik")

### 2a. Test retrieval → generation (langkah per langkah)

In [14]:
query = "apa hukuman untuk orang yang mencuri motor?"
print(f"Query: {query}\n")

# Step 1: Retrieval
print("--- Tahap Retrieval ---")
t0 = time.time()
docs = search(query, "hybrid", res, top_k=3, use_reranker=False, use_rewriting=True)
print(f"Ditemukan {len(docs)} pasal dalam {time.time()-t0:.1f} detik:")
for d in docs:
    print(f"  Pasal {d.metadata['pasal']} (skor: {d.metadata.get('score', '?'):.4f})")
    print(f"    {d.page_content[:100]}...")

# Step 2: Generation (tanpa verifikasi dulu)
print("\n--- Tahap Generation (tanpa verifikasi) ---")
t0 = time.time()
result = generate(query, docs, verify=False)
print(f"Selesai dalam {time.time()-t0:.1f} detik")
print(f"Sitasi: {result.cited_pasal}")
print(f"\nJawaban:\n{result.answer}")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Ditemukan 3 pasal dalam 31.0 detik:
  Pasal 476 (skor: 0.0328)
    Pasal 476

Setiap Orang yang mengambil suatu Barang yang sebagian atau seluruhnya milik orang lain, ...
  Pasal 486 (skor: 0.0323)
    Pasal 486
Setiap Orang yang secara melawan hukum memiliki suatu Barang yang sebagian atau seluruhnya...
  Pasal 542 (skor: 0.0308)
    Pasal 542

Setiap Orang yang menggunakan Kapal untuk menahan atau melakukan Kekerasan atau Ancaman K...

--- Tahap Generation (tanpa verifikasi) ---
Selesai dalam 58.4 detik
Sitasi: [476]

Jawaban:
Konteks tidak menyebutkan secara spesifik mengenai motor, namun untuk tindakan mengambil suatu barang yang sebagian atau seluruhnya milik orang lain dengan maksud untuk dimiliki secara melawan hukum (pencurian), hukumannya adalah pidana penjara paling lama 5 (lima) tahun atau pidana denda paling banyak kategori V [Pasal 476].


In [15]:
# Step 3: Generation DENGAN verifikasi
print("--- Tahap Generation (dengan verifikasi) ---")
t0 = time.time()
result_v = generate(query, docs, verify=True)
print(f"Selesai dalam {time.time()-t0:.1f} detik")
print(f"Didukung: {result_v.is_supported} ({result_v.verification.supported})")
print(f"Sitasi: {result_v.cited_pasal}")
print(f"Abstain: {result_v.abstained}")
if result_v.verification.unsupported_claims:
    print(f"Klaim tidak didukung: {result_v.verification.unsupported_claims}")
print(f"\n{format_answer(result_v)}")

--- Tahap Generation (dengan verifikasi) ---
Selesai dalam 17.6 detik
Didukung: True (ya)
Sitasi: [476]
Abstain: False

Orang yang melakukan pencurian, yaitu mengambil suatu barang yang sebagian atau seluruhnya milik orang lain dengan maksud untuk dimiliki secara melawan hukum, dipidana dengan pidana penjara paling lama 5 (lima) tahun atau pidana denda paling banyak kategori V [Pasal 476].

Dasar hukum: Pasal 476


### 2b. Test `ask()` end-to-end

In [16]:
pertanyaan = [
    "apa hukuman untuk orang yang mencuri motor?",
    "kalau nipu orang lewat olshop bisa dipenjara berapa tahun?",
    "pasal 300",
]

for q in pertanyaan:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    print(f"{'='*60}")
    t0 = time.time()
    r = ask(q, res=res, method="hybrid", top_k=3, use_rewriting=True, use_expansion=True, verify=True)
    dt = time.time() - t0
    print(f"⏱  {dt:.1f} detik | Didukung: {r.is_supported} | Pasal: {r.cited_pasal} | Abstain: {r.abstained}")
    print(f"\n{format_answer(r)}")


Q: apa hukuman untuk orang yang mencuri motor?
⏱  142.6 detik | Didukung: True | Pasal: [476] | Abstain: False

Konteks tidak menyebutkan secara spesifik mengenai motor, namun untuk tindakan mengambil suatu barang yang sebagian atau seluruhnya milik orang lain dengan maksud untuk dimiliki secara melawan hukum (pencurian), hukumannya adalah pidana penjara paling lama 5 (lima) tahun atau pidana denda paling banyak kategori V [Pasal 476].

Dasar hukum: Pasal 476

Q: kalau nipu orang lewat olshop bisa dipenjara berapa tahun?
⏱  92.8 detik | Didukung: True | Pasal: [492] | Abstain: False

Jika tindakan tersebut merupakan penipuan dengan menggunakan tipu muslihat atau rangkaian kata bohong untuk menggerakkan orang agar menyerahkan suatu barang, maka pelaku dapat dipidana dengan pidana penjara paling lama 4 (empat) tahun [Pasal 492].

Dasar hukum: Pasal 492

Q: pasal 300
⏱  32.4 detik | Didukung: True | Pasal: [300] | Abstain: False

Pasal 300 mengatur tentang perbuatan di muka umum yang dit

### 2c. Edge cases

In [17]:
# Test scope gate: pertanyaan bukan hukum
print("--- Scope gate: pertanyaan bukan hukum ---")
r = ask("1 tambah 1 berapa", res=res, use_scope_gate=True, verify=True)
print(f"Abstain: {r.abstained}")
print(f"Answer: {r.answer}")
assert r.abstained == True and r.status == "berhenti"
assert "di luar cakupan" in r.answer

--- Scope gate: pertanyaan bukan hukum ---
Abstain: True
Answer: Pertanyaan ini di luar cakupan chatbot KUHP.


In [18]:
# Test pasal tidak ada di korpus: pesannya harus "tidak ditemukan", dengan maupun tanpa scope gate
print("--- Pasal tidak ditemukan ---")
for gate in (False, True):
    r = ask("jelaskan pasal 9999", res=res, use_scope_gate=gate, verify=True)
    print(f"use_scope_gate={gate}: {r.status} | {r.answer}")
    assert r.status == "berhenti" and "9999 tidak ditemukan" in r.answer

--- Pasal tidak ditemukan ---
use_scope_gate=False: berhenti | Pasal 9999 tidak ditemukan dalam KUHP (UU No. 1 Tahun 2023).
use_scope_gate=True: berhenti | Pasal 9999 tidak ditemukan dalam KUHP (UU No. 1 Tahun 2023).


In [19]:
# Test lookup langsung (menyebut nomor pasal)
print("--- Lookup pasal langsung ---")
t0 = time.time()
r = ask("apa isi pasal 476?", res=res, verify=True)
print(f"⏱  {time.time()-t0:.1f} detik")
print(f"Didukung: {r.is_supported} | Pasal: {r.cited_pasal}")
print(f"\n{format_answer(r)}")

--- Lookup pasal langsung ---
⏱  27.8 detik
Didukung: True | Pasal: [476]

Isi Pasal 476 adalah setiap orang yang mengambil suatu barang yang sebagian atau seluruhnya milik orang lain, dengan maksud untuk dimiliki secara melawan hukum, dipidana karena pencurian [Pasal 476]. Pelaku dapat dijatuhi pidana penjara paling lama 5 (lima) tahun atau pidana denda paling banyak kategori V [Pasal 476].

Dasar hukum: Pasal 476


In [20]:
# --- Benchmark: waktu per tahap ---
import time

query = "apa hukuman untuk pencurian?"
docs = search(query, "hybrid", res, top_k=3, use_rewriting=True, use_expansion=True, use_reranker=True)

# Generate saja
t0 = time.time()
r1 = generate(query, docs, verify=False)
t_gen = time.time() - t0

# Generate + verify
t0 = time.time()
r2 = generate(query, docs, verify=True)
t_gen_v = time.time() - t0

print(f"Generate saja   : {t_gen:.1f} detik")
print(f"Generate+verify : {t_gen_v:.1f} detik")
print(f"Overhead verify : {t_gen_v - t_gen:.1f} detik")

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Generate saja   : 19.0 detik
Generate+verify : 13.5 detik
Overhead verify : -5.5 detik


In [21]:
# Jalur Abstain / Clarify dengan model asli. Hasilnya bergantung model: periksa manual, bukan assert.
uji = [
    ("ada jawabannya di KUHP", "apa hukuman untuk orang yang mencuri motor?"),
    ("hukum, tapi bukan KUHP", "berapa denda tidak memakai helm saat berkendara?"),
    ("ambigu", "saya dicolek orang di bus, bisa dipidana?"),
    ("pasal tidak ada", "apa isi pasal 9999?"),
]
for label, q in uji:
    t0 = time.time()
    r = ask(q, res=res, top_k=3, use_rewriting=True, use_expansion=True, use_scope_gate=True)
    print(f"\n[{label}] {q}")
    print(f"  status={r.status} | percobaan={r.attempts} | sitasi={r.cited_pasal} | {time.time()-t0:.0f} detik")
    print("  " + format_answer(r, show_verification=True).replace("\n", "\n  "))


[ada jawabannya di KUHP] apa hukuman untuk orang yang mencuri motor?
  status=jawab | percobaan=1 | sitasi=[476] | 147 detik
  Konteks tidak menyebutkan secara spesifik mengenai motor, namun untuk tindakan mengambil suatu barang yang sebagian atau seluruhnya milik orang lain dengan maksud untuk dimiliki secara melawan hukum (pencurian), hukumannya adalah pidana penjara paling lama 5 (lima) tahun atau pidana denda paling banyak kategori V [Pasal 476].
  
  Dasar hukum: Pasal 476
  
  --- Verifikasi ---
  Status: jawab | verdict: ya | percobaan: 1
  Raw: DIDUKUNG: ya
  KLAIM TIDAK DIDUKUNG: tidak ada
  PASAL YANG DIRUJUK: 476

[hukum, tapi bukan KUHP] berapa denda tidak memakai helm saat berkendara?
  status=abstain | percobaan=1 | sitasi=[] | 95 detik
  Mohon maaf, berdasarkan pasal-pasal KUHP (UU No. 1 Tahun 2023) yang ditemukan, bukti belum cukup untuk menjawab pertanyaan ini secara meyakinkan.
  
  Pasal yang ditemukan dan mungkin terkait (belum tentu menjawab pertanyaan):
  - Pasal